# ChickenStress — 04. Image Quality Gate (Inference / Produksi)
### Bagian 4 dari 4 (`01_image_processing.ipynb` → `02_models.ipynb` → `03_evaluate.ipynb` → `04_inference.ipynb`)

Notebook ini **terpisah dari alur training** (01–03) karena memang beda konteks: 01–03 memproses **dataset statis** untuk melatih & mengevaluasi model, sedangkan notebook ini berisi komponen **BAGIAN B** dari `alur_deteksi_stres_ayam.md` — Image Quality Gate yang jalan **saat model sudah dipasang di kandang**, memfilter frame dari kamera secara real-time sebelum masuk ke model.

Tidak ada dependency ke notebook 01–03 di sini kecuali contoh path dataset test untuk demo (opsional, bisa diganti ke stream kamera sungguhan).

In [1]:
import os
import cv2
import numpy as np
import pandas as pd
from collections import deque, Counter

# Ganti ke path dataset kerja kamu (hasil dari 01_image_processing.ipynb) kalau mau demo pakai contoh nyata,
# atau abaikan sepenuhnya kalau notebook ini langsung dihubungkan ke stream kamera produksi.
WORK_DIR = os.path.join(r"E:\Magang DSC\Code - Image Classification Workflow", "Data_working")
DEST = os.path.join(WORK_DIR, "dataset")

In [2]:
import cv2

def quality_score(img_bgr, blur_threshold=100, dark_thresh=40, bright_thresh=215):
    """[A] Quality Scoring -- dijalankan cepat, TANPA deep learning, di edge device
    sebelum citra dikirim ke model klasifikasi. Metrik sama seperti validasi Step 1
    (blur score / brightness), tapi fungsinya di sini bukan untuk *membuang dari
    dataset*, melainkan untuk *memutuskan aksi berikutnya* terhadap frame yang lewat."""
    if img_bgr is None:
        return {"ok": False, "issues": ["gagal_dibaca_atau_blank"]}

    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    laplacian_var = cv2.Laplacian(gray, cv2.CV_64F).var()
    brightness = float(gray.mean())
    is_blank = bool(gray.std() < 3)  # deteksi occlusion/blank frame sederhana

    issues = []
    if laplacian_var < blur_threshold:
        issues.append("blur")
    if brightness < dark_thresh:
        issues.append("terlalu_gelap")
    if brightness > bright_thresh:
        issues.append("terlalu_terang")
    if is_blank:
        issues.append("blank_or_occluded")

    return {
        "ok": len(issues) == 0,
        "laplacian_var": laplacian_var,
        "brightness": brightness,
        "is_blank": is_blank,
        "issues": issues,
    }

In [3]:
def enhance_image(img_bgr):
    """[C] Perbaikan ringan (lightweight enhancement) untuk frame yang gagal quality
    check tapi masih berpotensi diselamatkan: CLAHE (kontras lokal, terutama untuk
    citra gelap), denoising ringan, sharpening ringan.
    PENTING: enhancement ini HANYA diterapkan di inference, TIDAK menggantikan
    augmentasi training di Step 4 -- augmentasi training tetap dipertahankan supaya
    model sejak awal terbiasa dengan variasi kualitas, tidak bergantung pada
    perbaikan di sisi inference."""
    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l = clahe.apply(l)
    enhanced = cv2.cvtColor(cv2.merge((l, a, b)), cv2.COLOR_LAB2BGR)

    enhanced = cv2.fastNlMeansDenoisingColored(
        enhanced, None, h=5, hColor=5, templateWindowSize=7, searchWindowSize=21
    )

    sharpen_kernel = np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]])
    enhanced = cv2.filter2D(enhanced, -1, sharpen_kernel)

    return enhanced

In [4]:
quality_skip_log = []  # [D]: log frame yang di-skip -- umpan balik ke WP A (monitoring kesehatan perangkat IoT)

def image_quality_gate(image_path, camera_id="unknown", kandang_id="unknown", blur_threshold=100):
    """Orkestrasi [A] -> [B]/[C] -> [D] untuk satu frame.
    Return: (frame_siap_pakai: np.ndarray | None, status: 'ok' | 'enhanced' | 'skipped')"""
    img_bgr = cv2.imread(image_path)
    score = quality_score(img_bgr, blur_threshold=blur_threshold)

    if score["ok"]:
        return img_bgr, "ok"  # [B] lanjut ke model

    enhanced = enhance_image(img_bgr)
    score_after = quality_score(enhanced, blur_threshold=blur_threshold)
    if score_after["ok"]:
        return enhanced, "enhanced"  # [C] berhasil diperbaiki -> [B] lanjut ke model

    # [D] masih buruk setelah diperbaiki -> skip, catat untuk monitoring
    quality_skip_log.append({
        "image_path": image_path,
        "camera_id": camera_id,
        "kandang_id": kandang_id,
        "issues_before": score["issues"],
        "issues_after": score_after["issues"],
    })
    return None, "skipped"

In [5]:
# Demo Image Quality Gate pada beberapa citra contoh dari dataset/test
demo_dir = os.path.join(DEST, "test")
demo_files = []
for cls in sorted(os.listdir(demo_dir)):
    cls_folder = os.path.join(demo_dir, cls)
    demo_files += [os.path.join(cls_folder, f) for f in os.listdir(cls_folder)[:5]]

gate_results = []
for path in demo_files:
    _, status = image_quality_gate(path, camera_id="cam01", kandang_id="kandang01")
    gate_results.append({"file": os.path.basename(path), "status": status})

gate_results_df = pd.DataFrame(gate_results)
print("Ringkasan status Quality Gate:")
display(gate_results_df["status"].value_counts())
display(gate_results_df.head(10))

skip_log_df = pd.DataFrame(quality_skip_log)
print(f"\nFrame di-skip: {len(skip_log_df)} dari {len(demo_files)} contoh")
if len(skip_log_df) > 0:
    display(skip_log_df)

Ringkasan status Quality Gate:


status
ok    9
Name: count, dtype: int64

,file,status
0,not_stress_0003.jpg,ok
1,not_stress_0004.jpg,ok
2,not_stress_0016.jpg,ok
3,not_stress_0017.jpg,ok
4,not_stress_0018.jpg,ok
5,stress_0003.jpg,ok
6,stress_0004.jpg,ok
7,stress_0019.jpg,ok
8,stress_0020.jpg,ok



Frame di-skip: 0 dari 9 contoh


## Confidence Gate & Voting Temporal ([F])

Setelah frame lolos quality gate dan diinferensi oleh model ([E], lihat `03_evaluate.ipynb`), prediksi dengan confidence rendah **tidak langsung memicu alarm** -- sistem menunggu beberapa frame berikutnya dan memakai voting/agregasi temporal (mayoritas dari beberapa frame berturut-turut) sebelum status "berisiko" dikonfirmasi ke dashboard/WhatsApp. Ambang confidence idealnya dikalibrasi dari ECE/Brier score (lihat evaluasi kalibrasi di `03_evaluate.ipynb`).

In [6]:
from collections import deque, Counter

def make_temporal_voter(window_size=5, confidence_threshold=0.6, min_votes_for_alert=3):
    """[F] Confidence Gate & Voting Temporal.
    - confidence < confidence_threshold -> diabaikan, tunggu frame berikutnya (tidak masuk window).
    - Setelah window penuh, alarm dipicu hanya jika mayoritas window = 'stress'
      (>= min_votes_for_alert dari window_size), mengurangi false alarm akibat satu frame buruk."""
    window = deque(maxlen=window_size)

    def update(label, confidence):
        if confidence < confidence_threshold:
            return {"trigger_alert": False, "reason": "confidence_rendah_tunggu_frame_berikutnya"}

        window.append(label)
        if len(window) < window.maxlen:
            return {"trigger_alert": False, "reason": "menunggu_window_temporal_penuh"}

        vote_label, vote_count = Counter(window).most_common(1)[0]
        if vote_label == "stress" and vote_count >= min_votes_for_alert:
            return {"trigger_alert": True, "reason": f"mayoritas {vote_count}/{window.maxlen} frame = stress", "label": vote_label}
        return {"trigger_alert": False, "reason": f"mayoritas belum stress ({vote_count}/{window.maxlen})", "label": vote_label}

    return update

# Contoh pemakaian (label & confidence disimulasikan, bukan hasil inferensi sungguhan)
voter = make_temporal_voter()
for label, conf in [("stress", 0.55), ("stress", 0.72), ("not_stress", 0.8), ("stress", 0.9), ("stress", 0.85), ("stress", 0.95)]:
    print((label, conf), "->", voter(label, conf))

print("\nData hasil skip/low-confidence bisa dikurasi lebih lanjut sebagai bahan fine-tuning di Tahun 2 (WP D 3.5.5),")
print("karena mencerminkan kondisi nyata lapangan yang mungkin belum terwakili di data pelatihan awal.")

('stress', 0.55) -> {'trigger_alert': False, 'reason': 'confidence_rendah_tunggu_frame_berikutnya'}
('stress', 0.72) -> {'trigger_alert': False, 'reason': 'menunggu_window_temporal_penuh'}
('not_stress', 0.8) -> {'trigger_alert': False, 'reason': 'menunggu_window_temporal_penuh'}
('stress', 0.9) -> {'trigger_alert': False, 'reason': 'menunggu_window_temporal_penuh'}
('stress', 0.85) -> {'trigger_alert': False, 'reason': 'menunggu_window_temporal_penuh'}
('stress', 0.95) -> {'trigger_alert': True, 'reason': 'mayoritas 4/5 frame = stress', 'label': 'stress'}

Data hasil skip/low-confidence bisa dikurasi lebih lanjut sebagai bahan fine-tuning di Tahun 2 (WP D 3.5.5),
karena mencerminkan kondisi nyata lapangan yang mungkin belum terwakili di data pelatihan awal.
